# DeepBranchAI: Fine-tune on VESSEL12

Downloads the pretrained MitoEye weights, VESSEL12 training data, sets up nnU-Net, trains for 100 epochs, runs inference on volumes 21-23, and validates against expert annotations.

Click **Run All** — everything is automatic.

In [ ]:
# ============================================================
# CONFIGURATION — edit this one path, everything else is automatic
# ============================================================
BASE_DIR = 'F:/DeepBranchAI'     # Windows example
# BASE_DIR = '/home/user/DeepBranchAI'  # Linux example
# ============================================================

# Zenodo record base
ZENODO_BASE = 'https://zenodo.org/records/19363534/files'

# MitoEye pretrained weights (used as starting point for fine-tuning)
ZENODO_MITOEYE_WEIGHT  = f'{ZENODO_BASE}/DeepBranchAI_MitoEye_fold2.pth?download=1'
ZENODO_MITOEYE_PLANS   = f'{ZENODO_BASE}/DeepBranchAI_MitoEye_nnUNetPlans.json?download=1'
ZENODO_MITOEYE_DATASET = f'{ZENODO_BASE}/DeepBranchAI_MitoEye_dataset.json?download=1'

# VESSEL12 training data (25.2 GB)
ZENODO_TRAINING_URL = f'{ZENODO_BASE}/DeepBranchAI_VESSEL12_training.zip?download=1'

# Test volumes + annotations
ZENODO_TEST_URL = f'{ZENODO_BASE}/DeepBranchAI_demo_data.zip?download=1'
# ============================================================

In [ ]:
import sys, os, torch, functools
sys.path.insert(0, os.path.abspath('..'))

# Patch torch.load for PyTorch 2.6+ compatibility with nnU-Net checkpoints
_original_torch_load = torch.load
@functools.wraps(_original_torch_load)
def _patched_torch_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

from deepbranchai_utils import setup_environment, check_gpu, download_and_extract, install_weights

paths = setup_environment(BASE_DIR)
print()
gpu_ok = check_gpu()
if not gpu_ok:
    raise RuntimeError('CUDA GPU required for training. See install instructions.')

## Step 1: Download Pretrained MitoEye Weights

In [ ]:
import urllib.request
from pathlib import Path

print('--- MitoEye Pretrained Weights ---')

extract_dir = paths['weights'] / 'DeepBranchAI_Zenodo'
weight_dir  = extract_dir / 'DeepBranchAI_MitoEye_weights'
config_dir  = extract_dir / 'configs'
weight_dir.mkdir(parents=True, exist_ok=True)
config_dir.mkdir(parents=True, exist_ok=True)

files_to_download = [
    (ZENODO_MITOEYE_WEIGHT,  weight_dir / 'DeepBranchAI_MitoEye_fold2.pth'),
    (ZENODO_MITOEYE_PLANS,   config_dir / 'DeepBranchAI_MitoEye_nnUNetPlans.json'),
    (ZENODO_MITOEYE_DATASET, config_dir / 'DeepBranchAI_MitoEye_dataset.json'),
]

for url, dst in files_to_download:
    if not dst.exists():
        print(f'  Downloading {dst.name} ...')
        urllib.request.urlretrieve(url, str(dst))
        print(f'  Saved to {dst}')
    else:
        print(f'  Already downloaded: {dst.name}')

install_weights(
    extract_dir, paths['nnUNet_results'], paths['nnUNet_preprocessed'], paths['nnUNet_raw'],
    dataset_name='Dataset4005_Mitochondria',
    trainer_dir='nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres',
    weight_subdir='DeepBranchAI_MitoEye_weights',
    config_prefix='DeepBranchAI_MitoEye',
)

PRETRAINED_WEIGHTS = str(
    paths['nnUNet_results'] / 'Dataset4005_Mitochondria' /
    'nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres' / 'fold_2' / 'checkpoint_best.pth'
)
assert os.path.exists(PRETRAINED_WEIGHTS), f'Missing: {PRETRAINED_WEIGHTS}'
print(f'Pretrained weights: {PRETRAINED_WEIGHTS}')

## Step 2: Download VESSEL12 Training Data

In [ ]:
import shutil, json
import numpy as np
import nibabel as nib
import tifffile
import pandas as pd
import matplotlib.pyplot as plt
import gc

# Download training data (nnU-Net raw chunks + configs)
download_and_extract(ZENODO_TRAINING_URL, str(paths['data']), 'DeepBranchAI_VESSEL12_training.zip')

# Download test volumes + annotations
download_and_extract(ZENODO_TEST_URL, str(paths['data']), 'DeepBranchAI_demo_data.zip')

In [ ]:
# Install training chunks into nnU-Net raw directory
DS3005_RAW = paths['nnUNet_raw'] / 'Dataset3005_Mitochondria'
DS3005_RAW.mkdir(parents=True, exist_ok=True)

# Find extracted training data and copy to nnU-Net structure
training_root = None
for candidate in [
    paths['data'] / 'DeepBranchAI_VESSEL12_training',
    paths['data'] / 'Dataset3005_Mitochondria',
    paths['data'],
]:
    if (candidate / 'imagesTr').exists():
        training_root = candidate
        break

if training_root is None:
    # Search recursively
    for p in paths['data'].rglob('imagesTr'):
        training_root = p.parent
        break

if training_root is None:
    raise FileNotFoundError('Training data not found. Check ZENODO_TRAINING_URL.')

print(f'Training data found at: {training_root}')

# Copy to nnU-Net raw directory if not already there
for subdir in ['imagesTr', 'labelsTr']:
    src = training_root / subdir
    dst = DS3005_RAW / subdir
    if not dst.exists() or len(list(dst.glob('*.nii.gz'))) == 0:
        if src.exists():
            print(f'Copying {subdir}...')
            shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
        else:
            print(f'WARNING: {src} not found')
    else:
        print(f'{subdir} already installed ({len(list(dst.glob("*.nii.gz")))} files)')

# Copy dataset.json
ds_json_src = training_root / 'dataset.json'
ds_json_dst = DS3005_RAW / 'dataset.json'
if ds_json_src.exists() and not ds_json_dst.exists():
    shutil.copy2(str(ds_json_src), str(ds_json_dst))

n_images = len(list((DS3005_RAW / 'imagesTr').glob('*.nii.gz')))
n_labels = len(list((DS3005_RAW / 'labelsTr').glob('*.nii.gz')))
print(f'Training chunks: {n_images} images, {n_labels} labels')

## Step 3: Run nnU-Net Preprocessing

In [ ]:
from nnunetv2.experiment_planning.plan_and_preprocess_api import extract_fingerprints, plan_experiments, preprocess

DS3005_PREPROC = paths['nnUNet_preprocessed'] / 'Dataset3005_Mitochondria'
plans_file = DS3005_PREPROC / 'nnUNetPlans.json'

# Check if preprocessing already done
preproc_dir = DS3005_PREPROC / 'nnUNetPlans_3d_fullres'
if not preproc_dir.exists() or len(list(preproc_dir.glob('*.npz'))) < n_images:
    print('Running fingerprint extraction + experiment planning + preprocessing...')
    print('(this takes ~20 minutes)')

    extract_fingerprints([3005], num_processes=4, check_dataset_integrity=True)
    plan_experiments([3005])
    preprocess([3005], 'nnUNetPlans', configurations=['3d_fullres'], num_processes=[4])

    print('Preprocessing complete.')
else:
    print(f'Preprocessing already done ({len(list(preproc_dir.glob("*.npz")))} files)')

In [ ]:
# Modify plans: set patch_size to 352x352x128 and match Dataset4005 architecture
import json

with open(str(plans_file), 'r') as f:
    plans_3005 = json.load(f)

# Load Dataset4005 plans to copy architecture
plans_4005_path = paths['nnUNet_preprocessed'] / 'Dataset4005_Mitochondria' / 'nnUNetPlans.json'
if plans_4005_path.exists():
    with open(str(plans_4005_path), 'r') as f:
        plans_4005 = json.load(f)
    plans_3005['configurations']['3d_fullres']['architecture'] = plans_4005['configurations']['3d_fullres']['architecture'].copy()
    print('Copied architecture from Dataset4005')
else:
    # Use the architecture from Zenodo config
    config_path = paths['weights'] / 'DeepBranchAI_Zenodo' / 'configs' / 'DeepBranchAI_MitoEye_nnUNetPlans.json'
    with open(str(config_path), 'r') as f:
        plans_4005 = json.load(f)
    plans_3005['configurations']['3d_fullres']['architecture'] = plans_4005['configurations']['3d_fullres']['architecture'].copy()
    print('Copied architecture from Zenodo config')

plans_3005['configurations']['3d_fullres']['patch_size'] = [352, 352, 128]
plans_3005['configurations']['3d_fullres']['batch_size'] = 2

with open(str(plans_file), 'w') as f:
    json.dump(plans_3005, f, indent=4)

n_stages = plans_3005['configurations']['3d_fullres']['architecture']['arch_kwargs']['n_stages']
print(f'Plans updated: patch_size=[352,352,128], n_stages={n_stages}')

In [ ]:
# Install splits (volume-aware 5-fold cross-validation)
splits_src = None
for p in paths['data'].rglob('splits_final.json'):
    splits_src = p
    break

splits_dst = DS3005_PREPROC / 'splits_final.json'
if splits_src and splits_src.exists():
    shutil.copy2(str(splits_src), str(splits_dst))
    print(f'Installed volume-aware splits')
elif not splits_dst.exists():
    print('WARNING: splits_final.json not found - nnU-Net will use random splits')

## Step 4: Fine-tune (100 epochs)

In [ ]:
FOLD = 2

checkpoint = (
    paths['nnUNet_results'] / 'Dataset3005_Mitochondria' /
    'nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres' /
    f'fold_{FOLD}' / 'checkpoint_best.pth'
)

if not checkpoint.exists():
    from nnunetv2.run.run_training import run_training

    print('Starting fine-tuning (100 epochs, ~16 hours on RTX A6000)...')
    run_training(
        dataset_name_or_id='3005',
        configuration='3d_fullres',
        fold=FOLD,
        trainer_class_name='nnUNetTrainer_100epochs',
        pretrained_weights=PRETRAINED_WEIGHTS,
    )
    print('Training complete.')
else:
    print(f'Training already complete: {checkpoint}')

## Step 5: Inference on Test Volumes (21, 22, 23)

In [ ]:
TEST_VOLUMES = ['VESSEL12_21', 'VESSEL12_22', 'VESSEL12_23']
TMP_IN  = paths['data'] / 'tmp_input'
TMP_OUT = paths['data'] / 'tmp_output'

prob_maps = {}

# Create predictor once (reused across volumes)
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

predictor = nnUNetPredictor(
    tile_step_size=0.5,
    use_gaussian=True,
    use_mirroring=True,
    device=torch.device('cuda', 0),
    verbose=True,
    verbose_preprocessing=True,
    allow_tqdm=True,
)

model_folder = str(
    paths['nnUNet_results'] / 'Dataset3005_Mitochondria' /
    'nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres'
)
predictor.initialize_from_trained_model_folder(
    model_folder,
    use_folds=(FOLD,),
    checkpoint_name='checkpoint_best.pth',
)

for case_id in TEST_VOLUMES:
    # Find the raw test volume
    raw_path = None
    for p in paths['data'].rglob(f'{case_id}_raw.tif'):
        raw_path = p
        break
    if raw_path is None:
        print(f'WARNING: {case_id}_raw.tif not found, skipping')
        continue

    # Check for cached probability map
    npy_path = paths['data'] / f'{case_id}_prob.npy'
    if npy_path.exists():
        print(f'{case_id}: loading cached probability map')
        prob_maps[case_id] = np.load(str(npy_path))
        continue

    print(f'{case_id}: loading {raw_path.name}...')
    stack = tifffile.imread(str(raw_path))
    if stack.ndim == 4:
        stack = stack[..., 0]
    print(f'  Shape: {stack.shape}')

    # Prepare input
    for d in [TMP_IN, TMP_OUT]:
        if d.exists():
            shutil.rmtree(str(d))
        d.mkdir(parents=True)

    nib.save(
        nib.Nifti1Image(stack.astype(np.float32), np.eye(4)),
        str(TMP_IN / 'case_0000.nii.gz')
    )

    print(f'  Running inference...')
    predictor.predict_from_files(
        list_of_lists_or_source_folder=str(TMP_IN),
        output_folder_or_list_of_truncated_output_files=str(TMP_OUT),
        save_probabilities=True,
        overwrite=True,
        num_processes_preprocessing=1,
        num_processes_segmentation_export=1,
    )

    # Load probability map
    npz_files = list(TMP_OUT.glob('*.npz'))
    data = np.load(str(npz_files[0]))
    arr = data[data.files[0]]
    prob = arr[1] if arr.ndim == 4 else arr
    if prob.shape != stack.shape:
        prob = prob.transpose((2, 1, 0))

    prob_maps[case_id] = prob
    np.save(str(npy_path), prob)
    print(f'  Saved: {npy_path}')
    gc.collect()

print(f'\nInference complete for {len(prob_maps)} volumes.')

## Step 6: Validate Against Expert Annotations

In [ ]:
totals = dict(points=0, matched=0, vessel=0, vessel_matched=0, non_vessel=0, non_vessel_matched=0)
results = {}

for case_id, prob in prob_maps.items():
    # Find annotation CSV
    ann_path = None
    for p in paths['data'].rglob(f'{case_id}_Annotations.csv'):
        ann_path = p
        break
    if ann_path is None:
        print(f'{case_id}: annotations not found, skipping')
        continue

    df = pd.read_csv(str(ann_path), header=None, names=['x', 'y', 'z', 'label'])
    df[['x', 'y', 'z']] = df[['x', 'y', 'z']] * 2
    df = df.astype({'x': int, 'y': int, 'z': int, 'label': int})

    mask_bin = (prob >= 0.5).astype(np.uint8)
    pred = mask_bin[df['z'].values, df['y'].values, df['x'].values]
    true = df['label'].values

    n_pts      = len(true)
    n_match    = int((pred == true).sum())
    n_vess     = int((true == 1).sum())
    n_non      = int((true == 0).sum())
    n_vess_mat = int((pred[true == 1] == 1).sum())
    n_non_mat  = int((pred[true == 0] == 0).sum())

    results[case_id] = dict(
        points=n_pts, matched=n_match,
        vessel=n_vess, vessel_matched=n_vess_mat,
        non_vessel=n_non, non_vessel_matched=n_non_mat,
    )
    for k in totals:
        totals[k] += results[case_id].get(k, 0)

# Print results
print('Per-volume results:')
print('=' * 60)
rows = []
for case_id, s in results.items():
    acc = s['matched'] / s['points']
    v_acc = s['vessel_matched'] / s['vessel'] if s['vessel'] else 0
    n_acc = s['non_vessel_matched'] / s['non_vessel'] if s['non_vessel'] else 0
    print(f"  {case_id}: {acc:.2%} accuracy  (vessel {v_acc:.2%}, non-vessel {n_acc:.2%})")
    rows.append({'Volume': case_id, 'Accuracy': f'{acc:.2%}',
                 'Vessel Sens': f'{v_acc:.2%}', 'Non-vessel Spec': f'{n_acc:.2%}',
                 'Points': s['points']})

oa = totals['matched'] / totals['points']
ov = totals['vessel_matched'] / totals['vessel'] if totals['vessel'] else 0
on = totals['non_vessel_matched'] / totals['non_vessel'] if totals['non_vessel'] else 0
print(f"\n  Overall: {oa:.2%} accuracy  (vessel {ov:.2%}, non-vessel {on:.2%})")
rows.append({'Volume': 'Overall', 'Accuracy': f'{oa:.2%}',
             'Vessel Sens': f'{ov:.2%}', 'Non-vessel Spec': f'{on:.2%}',
             'Points': totals['points']})

display(pd.DataFrame(rows))

In [ ]:
# Visualize one test volume
case_id = 'VESSEL12_21'
if case_id in prob_maps:
    prob = prob_maps[case_id]
    mask = (prob >= 0.5).astype(np.uint8)
    
    raw_path = None
    for p in paths['data'].rglob(f'{case_id}_raw.tif'):
        raw_path = p
        break
    raw = tifffile.imread(str(raw_path))
    if raw.ndim == 4:
        raw = raw[..., 0]
    
    mid = raw.shape[0] // 2
    
    fig, axes = plt.subplots(3, 1, figsize=(18, 20))
    axes[0].imshow(raw[mid], cmap='gray')
    axes[0].set_title(f'{case_id} Raw CT (z={mid})', fontsize=20)
    axes[0].axis('off')
    
    axes[1].imshow(mask[mid], cmap='Reds', vmin=0, vmax=1)
    axes[1].set_title(f'Segmentation (z={mid})', fontsize=20)
    axes[1].axis('off')
    
    rgb = np.stack([raw[mid]] * 3, axis=-1).astype(np.float32) / 255
    v = mask[mid] > 0
    rgb[v, 0] = 1.0; rgb[v, 1] *= 0.3; rgb[v, 2] *= 0.3
    axes[2].imshow(rgb)
    axes[2].set_title(f'Overlay (z={mid})', fontsize=20)
    axes[2].axis('off')
    
    plt.suptitle(f'{case_id} — Fine-tuned DeepBranchAI', fontsize=22, y=1.0)
    plt.tight_layout()
    plt.show()